# Embed Products → Supabase (GPU)

Computes BGE-M3 embeddings for all products in **Supabase** and writes them back as
`halfvec(1024)`. Loads the fine-tuned model from the **HF Hub**.

**Before running:** Runtime → Change runtime type → **T4 GPU**. ~15–30 min on T4.
The HNSW index is built separately (by the team) after this completes.

In [ ]:
!pip install -q sentence-transformers psycopg2-binary huggingface_hub

## Configure

In [ ]:
# Paste your Supabase TRANSACTION-POOLER URL (port 6543) — the same DATABASE_URL
# from backend/.env (password already URL-encoded). Keep it private.
DATABASE_URL = "postgresql://postgres.<ref>:<ENCODED_PW>@aws-1-ap-southeast-2.pooler.supabase.com:6543/postgres"  # <-- PASTE

MODEL_REPO_ID = "kevin7548/chatbeauty-models"
MODEL_SUBDIR = "retrieval/bge-m3-finetuned-20260202-120852"
BATCH_SIZE = 256

## 1. Load model from the HF Hub

In [ ]:
from huggingface_hub import snapshot_download
from sentence_transformers import SentenceTransformer

local = snapshot_download(MODEL_REPO_ID, allow_patterns=f"{MODEL_SUBDIR}/*")
model = SentenceTransformer(f"{local}/{MODEL_SUBDIR}")
print("Model loaded. Device:", model.device)

## 2. Fetch products needing embeddings

In [ ]:
import psycopg2

conn = psycopg2.connect(DATABASE_URL)
cur = conn.cursor()
cur.execute("SELECT COUNT(*) FROM products")
print("total products:", cur.fetchone()[0])
cur.execute("""
    SELECT parent_asin, embedding_text
    FROM products
    WHERE embedding IS NULL AND embedding_text IS NOT NULL
    ORDER BY parent_asin
""")
rows = cur.fetchall()
cur.close(); conn.close()
asins = [r[0] for r in rows]
texts = [r[1] for r in rows]
print("to embed:", len(texts))

## 3. Encode (GPU)

In [ ]:
import time
start = time.time()
embs = model.encode(texts, batch_size=BATCH_SIZE, show_progress_bar=True, convert_to_numpy=True)
print(f"encoded {len(texts)} in {time.time()-start:.0f}s; shape {embs.shape}")

## 4. Write embeddings back (halfvec)

In [ ]:
from tqdm.auto import tqdm

conn = psycopg2.connect(DATABASE_URL)
cur = conn.cursor()
UPDATE_SQL = "UPDATE products SET embedding = %s::halfvec WHERE parent_asin = %s"
WRITE_BATCH = 500
for i in tqdm(range(0, len(asins), WRITE_BATCH), desc="writing"):
    data = [(str(e.tolist()), a) for a, e in zip(asins[i:i+WRITE_BATCH], embs[i:i+WRITE_BATCH])]
    cur.executemany(UPDATE_SQL, data)
    conn.commit()
cur.close(); conn.close()
print("wrote", len(asins), "embeddings")

## 5. Verify

In [ ]:
conn = psycopg2.connect(DATABASE_URL)
cur = conn.cursor()
cur.execute("SELECT COUNT(*) FROM products WHERE embedding IS NOT NULL")
print("products with embeddings:", cur.fetchone()[0])
cur.execute("SELECT pg_size_pretty(pg_database_size(current_database()))")
print("db size:", cur.fetchone()[0])
cur.close(); conn.close()
print("Done. Tell Claude — the HNSW index is built next.")